- Run codes: can you run the counterpart team's codes smoothly? So, are the results replicable?
- Data processing: Check the codes. Whether the machine learning project process the data correctly?
- Sample choice: Whether the machine learning project divide the sample correctly, does it have data leakage?
- List a least N suggestions in the machine learning projects(***IMPORTANT***, N = number of the teammates)
- Due 2 weeks later before class time

## 1.Run codes 

**Repetition**: The code contains repetitive logic. The preprocessing steps for the Price and Rent datasets are nearly identical. To streamline the code and simplify debugging, we can first standardize the column names across both datasets, then use a single, shared preprocessing function.

**Modularization**: Consolidating all code into one cell hampers execution and debugging, as a single error necessitates a full re-run and wastes time. It is advisable to modularize the code by dividing it into separate cells based on function.

**Performance**: On a standard business laptop, the total runtime is approximately one hour. The primary computational bottlenecks are the LASSO regression and the GridSearchCV hyperparameter tuning processes.

## 2.Data processing

There is a critical data leakage issue in the preprocessing stage:

In the function "clean_complex_features_optimized":
```python
for col in df.filter(like='_均值').columns:
    if df[col].isnull().any():
        median_val = df[col].dropna().median()
        if pd.isna(median_val): median_val = 0
        df[col] = df[col].fillna(median_val)
```

During data processing:
```python
all_data = pd.concat([train_data.drop(columns=[target]), test_data], keys=['train', 'test'])
all_data_cleaned = clean_complex_features_optimized(all_data)  
```

### (1) Test Set Information Leakage to Training Process:

The feature engineering operations (such as median imputation and regex extraction) are performed on the combined dataset containing both training and test samples. This means:

- Statistical Calculations Use Test Data: When computing median values for imputation in the clean_complex_features_optimized function, the calculations include test set samples, allowing information from future/unseen data to influence the training process

- Feature Extraction Contamination: Operations like extracting numerical ranges from text fields (extract_avg_value) use patterns from test data to determine how to process training features

- Data-Driven Decisions Compromised: Any data-dependent decisions made during cleaning (thresholds, transformation parameters) become biased by including test samples


### (2) Data Distribution Contamination: 
The statistical properties from the test set improperly influence training set preprocessing parameters:

- Distribution Shift Masking: By combining datasets, the true distribution differences between training and test populations become obscured, preventing the model from learning the actual training distribution

- Over-optimistic Performance: Models may appear to perform better during validation because they've indirectly "seen" patterns from the test domain during preprocessing

- Generalization Compromise: The trained model may fail to generalize to truly unseen data since it was exposed to test set characteristics during feature engineering

**Impact**: This leakage artificially inflates model performance metrics and creates false confidence in the model's generalization capability to new, truly unseen data.


## 3.Suggestions

**Suggestion 1: Fix Data Leakage in Feature Engineering**

The feature engineering process should be restructured to ensure that all transformations, imputations, and feature creations are learned exclusively from the training data. The clean_complex_features_optimized function should be applied separately to training and test sets, with all statistical parameters (medians, extraction patterns, transformation thresholds) calculated from the training set only and then applied to the test set without recalculation.

**Suggestion 2: Implement More Extensive Feature Engineering**

Consider developing a broader range of feature engineering techniques including interaction terms between important numerical features, temporal features if date information exists, clustering-based features using geographical coordinates, target encoding for high-cardinality categorical variables, and rolling statistics for sequential data patterns. All feature engineering should be fitted exclusively on training data.

**Suggestion 3: Adopt More Tolerant Outlier Handling Strategy**

Instead of completely removing outliers based on target variable IQR, consider using winsorization techniques that cap extreme values rather than eliminating them entirely. Additionally, extend outlier treatment to feature factors themselves using multivariate outlier detection methods that can identify anomalous patterns across multiple features simultaneously, preserving more data while reducing the influence of extreme values.

**Suggestion 4: Improve the code architecture**

First, you can try to split the code for processing a dataset into different cells based on functionality, modularizing the code to make it easier to run and debug. Then, to simplify the code, you can start by unifying the variable names for the Price and Rent datasets and then process them using the same preprocessing function. Finally, you can try to leverage the advantages of sparse matrices to reduce memory usage or decrease the maximum number of iterations of the model to save runtime.

**Suggestion 5:  Introduce Spatial Clustering Features to Address the Nonlinearity of Longitude and Latitude**

In Team 3's code, longitude (lon) and latitude (lat) are treated as standard numerical features, processed using log transformation (np.log1p) and standardization (StandardScaler).
Problem: The relationship between housing prices and geographical location is highly nonlinear. Linear models (OLS, Ridge, Lasso) cannot capture complex spatial relationships; they can only fit a simple plane.

Proposed Improvement:
K-Means Clustering: During the feature engineering stage, extract the (lon, lat) coordinates and use sklearn.cluster.KMeans to cluster all properties into N (e.g., N=20) "geographic clusters".Create New Feature: Add these N clusters as a categorical feature back into the dataset.

**Suggestion 6:  Use LassoCV and RidgeCV for Efficient Hyperparameter Tuning**

In Team 3's model, the param_grids are very small. For example, both Lasso and Ridge only test 3 alpha values.
Problem: GridSearchCV is slow when searching across a large range, which led to a limited search scope and most likely missed the optimal alpha value.

Suggestion:
Replace with LassoCV and RidgeCV: sklearn.linear_model provides LassoCV and RidgeCV. These estimators use a "regularization path" algorithm, which allows for extremely efficient cross-validation across a large number of alpha values to find a near-"continuous" optimal alpha.